# Bakehouse Data Ingestion to Bronze Layer

## Overview
This notebook ingests all tables from the `samples.bakehouse` schema into the bronze layer.



## Configuration
* **Widget**: `catalog_name` (default: "workspace") - The target catalog for bronze tables
* **Target Schema**: `{catalog_name}.bronze_raw`
* **Source Schema**: `samples.bakehouse`

## Tables Ingested
This notebook ingests the following 6 tables:
1. `media_customer_reviews` → `bakehouse_media_customer_reviews`
2. `media_gold_reviews_chunked` → `bakehouse_media_gold_reviews_chunked`
3. `sales_customers` → `bakehouse_sales_customers`
4. `sales_franchises` → `bakehouse_sales_franchises`
5. `sales_suppliers` → `bakehouse_sales_suppliers`
6. `sales_transactions` → `bakehouse_sales_transactions`

## Process
Each table is:
1. Read from the source using `spark.table()`
2. Row count is captured before writing
3. Written to the bronze layer with `overwrite` mode
4. Prefixed with `bakehouse_` to identify the source schema

In [0]:
# Create widget for catalog name
dbutils.widgets.text("catalog_name", "workspace", "Catalog Name")
catalog_name = dbutils.widgets.get("catalog_name")

In [0]:
# Create bronze_raw schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.bronze_raw")

In [0]:
from common.table_utils import get_source_table, get_target_table

def ingest_table(table_name):
    """
    Ingest a table from samples.bakehouse into the bronze layer.
    
    Args:
        table_name: Name of the table in samples.bakehouse schema
    
    Returns:
        int: Number of rows ingested
    """
    source_table = get_source_table("bakehouse", table_name)
    target_table = get_target_table(catalog_name, "bronze_raw", "bakehouse", table_name)
    
    # Read from source using spark.sql
    df = spark.sql(f"SELECT * FROM {source_table}")
    row_count = df.count()
    
    # Write to target
    df.write.mode("overwrite").saveAsTable(target_table)
    
    return row_count

In [0]:
# Array of table names to ingest from samples.bakehouse
table_names = [
    "media_customer_reviews",
    "media_gold_reviews_chunked",
    "sales_customers",
    "sales_franchises",
    "sales_suppliers",
    "sales_transactions"
]

In [0]:
# Loop through each table and ingest
for table_name in table_names:
    row_count = ingest_table(table_name)
    print(f"Ingested {table_name}: {row_count} rows")

print(f"\nCompleted ingestion of {len(table_names)} tables from samples.bakehouse to {catalog_name}.bronze_raw")

In [0]:
%sql
SHOW TABLES IN IDENTIFIER(:catalog_name).bronze_raw